# Bronze layer ingestion - load raw CVS's from Dunnhumby dataset into Delta tables

## Set context
Setting the default catalog and schema so later quaries don't need fully qualified table names.

In [0]:
%sql
USE CATALOG dunnhumby;
USE SCHEMA bronze;

## Load helper functions

Importing the row count sanity check from a separate module. Kept outside the notebook since it will be reused in silver and gold too.

In [0]:
from utils.data_quality import check_table_row_counts

## Ingest raw CSVs into bronze tables
For every CSV file in the raw_files volume, a matching Delta table is created in the bronze schema, one to one, with no transformations, following the bronze layer's rule.

In [0]:
def create_table_form_csv(file_name, table_name, columns="*"):
    path = f"/Volumes/dunnhumby/bronze/raw_files/{file_name}"
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS dunnhumby.bronze.{table_name} AS 
        SELECT {columns}
        FROM read_files(
            '{path}',
            format => 'csv',
            header => true,
            inferSchema => true
        )
    """)

files_to_tables = {
    "campaign_desc.csv": "campaign_descriptions",
    "campaign_table.csv": "campaign_table",
    "causal_data.csv": "causal_data",
    "coupon.csv": "coupon",
    "coupon_redempt.csv": "coupon_redemptions",
    "hh_demographic.csv": "household_demographic",
    "product.csv": "product",
    "transaction_data.csv": "transaction_data"
}

for file_name, table_name in files_to_tables.items():
    create_table_form_csv(file_name,table_name)

## Verify tables were created

In [0]:
%sql
SHOW TABLES IN dunnhumby.bronze

## Sanity check: row counts
Counting rows in every table and flagging any that came out empty. If ingestion failed to load something correctly (wrong delimiter, encoding issue), it shows up here right away instead of only surfacing later in silver.

In [0]:
display(check_table_row_counts("dunnhumby","bronze"))

##Summary
The bronze layer now holds 8 tables loaded 1:1 from the Dunnhumby CSV files. Next step: cleaning and standardization in the silver layer (02_silver_transform).